In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

In [17]:
PROJECT_ROOT = Path("D:/Dev/enterprise-aws-data-lakehouse-ml-system")
DATA_CURATED = PROJECT_ROOT / "data" / "curated"

train = pd.read_parquet(DATA_CURATED / "train_curated.parquet")

print(train.shape)

(590540, 450)


In [18]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,uid,uid_next_dt,uid_prev_dt,uid_time_to_next,uid_time_from_prev,uid_txn_count,uid_amt_mean,uid_amt_std,uid_amt_median,uid_amt_deviation
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,13926_315.0,1289868.0,NaN,1203468.0,NaN,3,142.000000,152.654021,68.50,-0.481481
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,2755_325.0,499543.0,NaN,413142.0,NaN,61,275.521639,609.754050,108.50,-0.404297
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,4663_330.0,951007.0,NaN,864538.0,NaN,34,52.433529,22.055991,49.00,0.297718
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,18132_476.0,94997.0,NaN,8498.0,NaN,257,124.769533,248.511730,88.50,-0.300869
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,4497_420.0,NaN,NaN,NaN,NaN,1,50.000000,NaN,50.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,6550_272.0,NaN,15293291.0,NaN,517756.0,21,63.945238,24.885583,59.00,-0.600558
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,10444_204.0,NaN,15684811.0,NaN,126238.0,12,242.041667,98.117383,280.00,-2.064279
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,12037_231.0,NaN,15811007.0,NaN,72.0,497,117.862435,145.583525,67.95,-0.596994
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,7826_387.0,NaN,15810836.0,NaN,252.0,815,141.874761,219.920764,100.00,-0.113108


In [19]:
"day" in train

True

### Create Time-Based Split

In [20]:
max_day = train["day"].max()
cutoff_day = max_day - 30  # last 30 days as validation

train_set = train[train["day"] <= cutoff_day]
val_set = train[train["day"] > cutoff_day]

print(train_set.shape, val_set.shape)

(505110, 450) (85430, 450)


### Define Target & Features

In [21]:
TARGET = "isFraud"

drop_cols = [
    "TransactionID",
    "uid",
    "uid_next_dt",
    "uid_prev_dt"
]

FEATURES = [
    col for col in train.columns
    if col not in drop_cols and col != TARGET
]

print("Total features:", len(FEATURES))

Total features: 445


### Create Train / Validation Splits

In [26]:
X_train = train_set[FEATURES].copy()
y_train = train_set[TARGET].copy()

X_val = val_set[FEATURES].copy()
y_val = val_set[TARGET].copy()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (505110, 445)
X_val shape: (85430, 445)


### Fraud Rate Over Time

In [22]:
fraud_by_day = train.groupby("day")["isFraud"].mean()      # If fraud rate changes drastically → instability risk.
fraud_by_day.describe()           

count    182.000000
mean       0.036048
std        0.009984
min        0.010972
25%        0.028577
50%        0.035639
75%        0.042467
max        0.069942
Name: isFraud, dtype: float64

### Feature Drift Check (Mean Shift)

In [23]:
feature = "uid_amt_deviation"

print("Train mean:", train_set[feature].mean())      # If difference is very large → drift problem.
print("Val mean:", val_set[feature].mean())

Train mean: 0.001339061819354304
Val mean: -0.007872886014686822


In [24]:
train.groupby("isFraud")["uid_amt_deviation"].mean()     # If fraud mean ≠ non-fraud mean → feature has signal.

isFraud
0   -0.005174
1    0.142609
Name: uid_amt_deviation, dtype: float64

### Detect Potential Leakage

In [25]:
train.corr(numeric_only=True)["isFraud"].sort_values(ascending=False).head(10)

isFraud    1.000000
V257       0.383060
V246       0.366878
V244       0.364129
V242       0.360590
V201       0.328005
V200       0.318783
V189       0.308219
V188       0.303582
V258       0.297151
Name: isFraud, dtype: float64

### Save Split Artifacts

In [27]:
SPLIT_PATH = PROJECT_ROOT / "data" / "splits"
SPLIT_PATH.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(SPLIT_PATH / "X_train.parquet", index=False)
X_val.to_parquet(SPLIT_PATH / "X_val.parquet", index=False)
y_train.to_frame().to_parquet(SPLIT_PATH / "y_train.parquet", index=False)
y_val.to_frame().to_parquet(SPLIT_PATH / "y_val.parquet", index=False)

print("Splits saved successfully to:", SPLIT_PATH)

Splits saved successfully to: D:\Dev\enterprise-aws-data-lakehouse-ml-system\data\splits
